In [ ]:
import os
import google.generativeai as genai
import json
import csv


API_KEY = os.environ['GEMINI_API_KEY']

# Initialize the Gemini client
genai.configure(api_key=API_KEY)
model = genai.GenerativeModel('gemini-2.0-flash')

c:\Users\Pc\anaconda3\envs\aiHomework1\Lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def read_csv(filepath):
    """Reads CSV and returns list of rows (as lists of strings)."""
    with open(filepath, newline='', encoding='utf-8') as csvfile:
        reader = csv.reader(csvfile)
        rows = list(reader)
    return rows

def read_script(filepath):
    """Reads the script file and returns its content as a string."""
    with open(filepath, 'r', encoding='utf-8') as file:
        return file.read()

def write_relationships_csv(filepath, relationships):
    """Writes relationship triples to CSV."""
    with open(filepath, "w", newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["source", "relationship", "target"])  # Header
        writer.writerows(relationships)

In [ ]:
def extract_relationships(script_text, characters, locations):
    """
    Build a prompt to extract relationships from the script using character and location data.
    Returns parsed relationship triples.
    """
    # Format character data for the prompt
    char_list = "\n".join([f"{i+1}. {row[0]} (normalized: {row[1]}, mentions: {row[2]})" 
                          for i, row in enumerate(characters[1:])])  # Skip header
    
    # Format location data for the prompt
    loc_list = "\n".join([f"{i+1}. {row[0]} (normalized: {row[1]}, mentions: {row[2]})" 
                         for i, row in enumerate(locations[1:])])  # Skip header
    
    # Create the prompt
    prompt = (
        f"You are an expert in analyzing film scripts and extracting relationships between entities. "
        f"I will provide you with a film script, a list of major characters, and a list of important locations. "
        f"Your task is to extract relationships between: \n"
        f"1. Character-to-Character relationships (e.g., 'John is friends with Mary', 'Alex is father of Bob')\n"
        f"2. Character-to-Location relationships (e.g., 'John visits Paris', 'Mary lives in London')\n\n"
        
        f"Characters:\n{char_list}\n\n"
        f"Locations:\n{loc_list}\n\n"
        
        f"Here is part of the script (if it's truncated, focus on what's available):\n\n"
        f"{script_text[:50000]}\n\n"  # Limit script length to avoid token limits
        
        f"Output the relationships as triples in the format 'Entity1,Relationship,Entity2' (one per line). "
        f"Use the normalized character/location names in your output. "
        f"Focus on extracting the 15-20 most important relationships that are clearly supported by the script. "
        f"Do not invent relationships that aren't evident in the text."
    )
    
    # Call Gemini API
    print("Sending relationship extraction prompt to Gemini API...")
    response = model.generate_content(prompt)
    response_text = response.text
  
    response_text = response_text.replace("```", "").strip()
    
    # Parse the relationships
    relationships = []
    for line in response_text.splitlines():
        line = line.strip()
        if line and "," in line:
            parts = line.split(",")
            if len(parts) >= 3:
                source = parts[0].strip()
                relationship = parts[1].strip()
                target = parts[2].strip()
                relationships.append([source, relationship, target])
    
    return relationships

In [ ]:
def create_knowledge_graph_visualization(characters, locations, relationships, output_file="knowledge_graph.html"):
    """
    Creates a simple HTML visualization of the knowledge graph using D3.js
    """
    nodes = []
    node_ids = set()  # Track existing node IDs
    
    # Add character nodes
    for i, char in enumerate(characters[1:]):  # Skip header
        node_ids.add(char[1])
        nodes.append({"id": char[1], "name": char[0], "type": "character"})
    
    # Add location nodes
    for i, loc in enumerate(locations[1:]):  # Skip header
        node_ids.add(loc[1])
        nodes.append({"id": loc[1], "name": loc[0], "type": "location"})
    
    # Collect all unique entities from relationships
    for rel in relationships:
        source = rel[0]
        target = rel[2]
        
        # Add missing sources or targets as "other" type nodes
        if source not in node_ids:
            node_ids.add(source)
            nodes.append({"id": source, "name": source, "type": "other"})
        
        if target not in node_ids:
            node_ids.add(target)
            nodes.append({"id": target, "name": target, "type": "other"})
    
    # Create links from relationships
    links = []
    for rel in relationships:
        links.append({
            "source": rel[0],
            "target": rel[2],
            "relationship": rel[1]
        })
    
    # Create a simple HTML template with D3.js
    html = """
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <title>Script Knowledge Graph</title>
        <script src="https://d3js.org/d3.v7.min.js"></script>
        <style>
            body { margin: 0; font-family: Arial, sans-serif; }
            .links line { stroke: #999; stroke-opacity: 0.6; }
            .nodes circle { stroke: #fff; stroke-width: 1.5px; }
            .character { fill: #66c2a5; }
            .location { fill: #fc8d62; }
            .other { fill: #8da0cb; }  /* New color for other entities */
            .text { font-size: 10px; }
        </style>
    </head>
    <body>
        <svg width="960" height="600"></svg>
        <script>
            // Graph data
            const graph = {
                "nodes": NODES_JSON,
                "links": LINKS_JSON
            };
            
            const svg = d3.select("svg"),
                width = +svg.attr("width"),
                height = +svg.attr("height");
            
            // Create simulation
            const simulation = d3.forceSimulation()
                .nodes(graph.nodes)
                .force("link", d3.forceLink(graph.links).id(d => d.id).distance(100))
                .force("charge", d3.forceManyBody().strength(-300))
                .force("center", d3.forceCenter(width / 2, height / 2));
            
            // Create links
            const link = svg.append("g")
                .attr("class", "links")
                .selectAll("line")
                .data(graph.links)
                .enter().append("line");
            
            // Create nodes
            const node = svg.append("g")
                .attr("class", "nodes")
                .selectAll("g")
                .data(graph.nodes)
                .enter().append("g");
            
            node.append("circle")
                .attr("r", 8)
                .attr("class", d => d.type)
                .call(d3.drag()
                    .on("start", dragstarted)
                    .on("drag", dragged)
                    .on("end", dragended));
            
            node.append("text")
                .attr("class", "text")
                .attr("dx", 12)
                .attr("dy", ".35em")
                .text(d => d.name);
            
            // Add relationship labels
            svg.append("g")
                .attr("class", "texts")
                .selectAll("text")
                .data(graph.links)
                .enter().append("text")
                .attr("class", "text")
                .attr("fill", "#666")
                .text(d => d.relationship);
            
            // Update positions
            simulation.on("tick", () => {
                link
                    .attr("x1", d => d.source.x)
                    .attr("y1", d => d.source.y)
                    .attr("x2", d => d.target.x)
                    .attr("y2", d => d.target.y);
                
                node.attr("transform", d => `translate(${d.x},${d.y})`);
                
                svg.selectAll(".texts text")
                    .attr("x", d => (d.source.x + d.target.x) / 2)
                    .attr("y", d => (d.source.y + d.target.y) / 2);
            });
            
            function dragstarted(event, d) {
                if (!event.active) simulation.alphaTarget(0.3).restart();
                d.fx = d.x;
                d.fy = d.y;
            }
            
            function dragged(event, d) {
                d.fx = event.x;
                d.fy = event.y;
            }
            
            function dragended(event, d) {
                if (!event.active) simulation.alphaTarget(0);
                d.fx = null;
                d.fy = null;
            }
        </script>
    </body>
    </html>
    """
    
    html = html.replace("NODES_JSON", json.dumps(nodes))
    html = html.replace("LINKS_JSON", json.dumps(links))
    
    # Write to file
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(html)
    
    print(f"Knowledge graph visualization created at {output_file}")
    
    # In a notebook, you may want to display the HTML directly
    from IPython.display import HTML, display
    display(HTML(output_file))


In [ ]:
SCRIPT_FILE = "film_script2.txt"  
CHARACTERS_CSV = "filtered_characters.csv"
LOCATIONS_CSV = "filtered_locations.csv"
OUTPUT_RELATIONSHIPS = "relationships.csv"

# Read input files
script_text = read_script(SCRIPT_FILE)
characters = read_csv(CHARACTERS_CSV)
locations = read_csv(LOCATIONS_CSV)

relationships = extract_relationships(script_text, characters, locations)

write_relationships_csv(OUTPUT_RELATIONSHIPS, relationships)
print(f"Extracted {len(relationships)} relationships and saved to {OUTPUT_RELATIONSHIPS}")


# create_knowledge_graph_visualization(characters, locations, relationships)

Sending relationship extraction prompt to Gemini API...
Extracted 16 relationships and saved to relationships.csv


In [8]:
create_knowledge_graph_visualization(characters, locations, relationships)

Knowledge graph visualization created at knowledge_graph.html
